# HDRL: how to run an instance

Run the cells in order. For a new run you normally only edit **`INSTANCE_ID`** (first setting of the instance cell); everything else has working defaults.

| Setting | What it does |
|---|---|
| `INSTANCE_ID` | Paper instance to train (table below). It selects the problem (`1.x` = S-CLSP, Case I; `2.x` = S-CLSP-SD, Case II), builds the instance and picks the matching environment and trainer. |
| `WARM_START` | `True`: pretrain the HIGH network with the AMBS heuristic before PPO. The result is cached in `runs/<exp_name>/high_warmstart.pt` and reused in later runs. |
| `WARM_START_FORCE_RETRAIN` | `True`: ignore that cache and pretrain again. |
| `HIDDEN_SIZES*` | MLP widths of HIGH / LOW (default `(256, 256)`). |
| `ENV_*` | Overrides of the environment helpers. `None` = automatic by K. |
| `LOW_RETURN_H`, `LOW_SUCC_H`, `LOW_REWARD_SCALE` | Window and scale of the LOW return. |

The notebook has three parts:

1. **Instance and hyperparameters**: builds the instance (`HYP`) from `INSTANCE_ID`. Always run it.
2. **Train**: trains HIGH + LOW with PPO. Skip it to test a model that is already trained.
3. **Test**: evaluates `high_joint_best.pt` / `low_joint_best.pt` over 10 000 trajectories. They are read from the same folder as this notebook, so no path is needed. If they are not there, it uses the ones just saved by the training cell in `runs/<exp_name>`.

To test a trained model: copy its two `.pt` files next to this notebook, set the same `INSTANCE_ID` and settings it was trained with, run the instance cell, and go straight to the test cell.
Seeds: train 123, validation 10000, test 12.

## Instance IDs

All instances use mu = 4, h = 1, b = 9, unit batch size. Demand **low** = U{3,5}, **high** = U{0,8}; **cf** is the capacity factor. Inside every block of four IDs the order is: (cf 1.1, low), (cf 1.5, low), (cf 1.1, high), (cf 1.5, high).

**Case I, S-CLSP (`1.1` – `1.24`)**

| K | IDs |
|---|---|
| 5 | 1.1 – 1.4 |
| 10 | 1.5 – 1.8 |
| 15 | 1.9 – 1.12 |
| 20 | 1.13 – 1.16 |
| 25 | 1.17 – 1.20 |
| 30 | 1.21 – 1.24 |

**Case II, S-CLSP-SD (`2.1` – `2.16`)**, sequence-dependent setups. Type A: two families. Type B: five families on a line.

| Type | K | IDs |
|---|---|---|
| A | 15 | 2.1 – 2.4 |
| A | 20 | 2.5 – 2.8 |
| B | 15 | 2.9 – 2.12 |
| B | 20 | 2.13 – 2.16 |

Example: `"1.19"` = K 25, cf 1.1, high demand; `"2.14"` = Type B, K 20, cf 1.5, low demand.


## 1. Instance and hyperparameters


In [ ]:
"""
Instance and hyperparameters of one benchmark instance of the paper (HDRL).

Set INSTANCE_ID (see the guide at the top of the notebook) and run. The ID selects the problem
(Case I: S-CLSP, Case II: S-CLSP-SD) and builds the instance and all the hyperparameters (HYP).
This cell does not train anything: training is the next cell and testing the last one. To test an
already trained model, run only this cell and the test cell.
Checkpoints, curves and the warm-start cache of a training run are written to HYP["out_dir"] = runs/<exp_name>.
"""
# 0) User settings
INSTANCE_ID = "1.5"

# MLP hidden sizes (None keeps the default (256, 256)); HIGH and LOW can be set separately.
HIDDEN_SIZES = None
HIDDEN_SIZES_HIGH = (256, 256)
HIDDEN_SIZES_LOW = (256, 256)

# Case I environment helpers (CLSPHierEnv.__init__). None = automatic by K, True/False = force.
#   K <= 10: only enable_Ilim and enable_ip_eligibility are on.
#   K > 10:  all on.
ENV_RESERVE_FUTURE_MIN = None
ENV_FORCE_MIN_BATCH = None
ENV_ENABLE_ILIM = None
ENV_ENFORCE_EOQ_QMAX = None
ENV_ENABLE_IP_ELIGIBILITY = None

# AMBS warm start of the HIGH network (behavior cloning before PPO; LOW starts at random).
# Thresholds come from AMBS_PARAMS_REGISTRY (warm_start.py) by INSTANCE_ID.
# The pretrained weights are cached in HYP["out_dir"]/high_warmstart.pt and reused in later runs.
WARM_START = True

# Warm-start options (omitted keys use the defaults of warm_start.py):
#   dataset_periods (AMBS periods simulated), rollouts (10), warmup_periods (100), epochs (8),
#   lr (3e-4), batch_size, entropy_coef, value_coef (0.5),
#   rank_coef (S-CLSP-SD only: weight of the Plackett-Luce order term, 1.0).
# AMBS periods simulated for the warm-start dataset, by instance size.
if INSTANCE_ID in ("1.1", "1.2", "1.3", "1.4"):
    WARM_START_DATASET_PERIODS = 100_000
elif INSTANCE_ID in ("1.5", "1.6", "1.7", "1.8"):
    WARM_START_DATASET_PERIODS = 150_000
elif INSTANCE_ID in ("1.9", "1.10", "1.11", "1.12"):
    WARM_START_DATASET_PERIODS = 200_000
elif INSTANCE_ID in ("1.13", "1.14", "1.15", "1.16"):
    WARM_START_DATASET_PERIODS = 250_000
elif INSTANCE_ID in ("1.17", "1.18", "1.19", "1.20"):
    WARM_START_DATASET_PERIODS = 300_000
elif INSTANCE_ID in ("1.21", "1.22", "1.23", "1.24"):
    WARM_START_DATASET_PERIODS = 350_000
# Case II (S-CLSP-SD): same size rule as Case I by K
elif INSTANCE_ID in ("2.1", "2.2", "2.3", "2.4", "2.9", "2.10", "2.11", "2.12"):   # K = 15
    WARM_START_DATASET_PERIODS = 200_000
elif INSTANCE_ID in ("2.5", "2.6", "2.7", "2.8", "2.13", "2.14", "2.15", "2.16"):  # K = 20
    WARM_START_DATASET_PERIODS = 250_000
else:
    raise ValueError(f"Unknown INSTANCE_ID '{INSTANCE_ID}' (Case I: 1.1..1.24, Case II: 2.1..2.16).")
WARM_START_KWARGS = {"dataset_periods": WARM_START_DATASET_PERIODS, "epochs": 300, "lr": 1e-4, "entropy_coef": 2e-2}

# True: ignore the cached high_warmstart.pt and pretrain again.
WARM_START_FORCE_RETRAIN = False

# LOW return (buffers.compute_low_window_returns): per-product Monte-Carlo return over a finite window.
#   LOW_RETURN_H      window (periods) of the product's own future cost.
#   LOW_SUCC_H        window of the future cost of the products visited after it in the period (0 disables).
#   LOW_REWARD_SCALE  the returns are divided by this constant.
# The last max(LOW_RETURN_H, LOW_SUCC_H) periods of each episode are not used to train LOW.
# Per-level gamma and entropy_coef are set in build_base_hyp.
LOW_RETURN_H = 20
LOW_SUCC_H = 20
LOW_REWARD_SCALE = 1000.0


# 1) Shared imports
import os
import random
import numpy as np
import torch


# 2) Shared reproducibility utilities
def set_global_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"

    np.random.seed(seed)
    random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


# 3) Instance builders (S-CLSP)
def build_base_hyp(K):
    """Base hyperparameter dictionary shared by all instances.

    Instance fields (capacity_factor, mu, bs, h, b, k, st, cov) are None and are filled by the instance
    builders.

    :param K: Number of products.
    :return: HYP dict (PPO, learning-rate schedule, logging and evaluation settings).
    """
    HYP = {

        # (A) Problem definition — CLSP instance parameters
        "K": K,                   # number of products
        "capacity_factor": None,  # capacity scaling factor
        "mu": None,               # mean demand vector
        "bs": None,               # batch size per product
        "h": None,                # holding cost
        "b": None,                # backorder cost
        "k": None,                # setup cost
        "st": None,               # setup times
        "cov": None,              # demand coefficient of variation


        # (B) Environment / rollout configuration
        "horizon": 256,           # episode length
        "steps_per_update": 1024, # rollout size per PPO update


        # (C) PPO core hyperparameters
        "updates": 8000,          # total PPO updates
        "gamma": 0.99,            # discount factor (shared default)
        "gamma_high": 0.99,       # discount factor override (HIGH)
        "gamma_low": 0.99,        # discount factor override (LOW)
        "lam": 0.95,              # GAE parameter
        "clip_ratio": 0.2,        # PPO clipping parameter
        "train_iters": 10,        # gradient steps per update
        "batch_size": 256,        # minibatch size
        "entropy_coef": 2e-2,     # entropy regularization (shared default)
        "entropy_coef_high": 2e-2,  # entropy regularization override (HIGH)
        "entropy_coef_low": 2e-2,   # entropy regularization override (LOW)
        "kl_target": 0.08,        # KL divergence threshold


        # (D) Learning-rate schedule (stepwise decay)
        "lr_high": 5e-5,          # initial LR (high-level policy)
        "lr_low":  5e-5,          # initial LR (low-level policy)
        "lr_high_levels": [5e-5, 3e-5, 1e-5],
        "lr_low_levels":  [5e-5, 3e-5, 1e-5],
        "lr_drop_patience_evals": 7,  # non-improving eval tolerance

        # (E) Logging, evaluation and early stopping
        "print_every": 10,
        "out_dir": ".",
        "use_wandb": False,
        "eval_seed_base": 10000,
        "early_stop_patience": 100,
    }

    return HYP


def build_hyp_set1(K, cov_level, cf):
    """Case I instance (homogeneous demand, as in van Hezewijk et al., 2025).

    mu = 4, h = 1, b = 9, TBO = 10 (k = h * TBO^2 * mu / 2), no setup times, unit batch size.
    Demand: cov 0.14 ("low", U{3,5}) or 0.58 ("high", U{0,8}).
    Initial state: I_0 = ceil(mu + 1.282 * sigma) with sigma = 1 (low) or 4 (high); machine set up for
    product 0.

    :param K: Number of products.
    :param cov_level: "low" or "high".
    :param cf: Capacity factor (1.1 or 1.5).
    """
    # Basic validation of inputs
    assert cov_level in ("low", "high")
    assert cf in (1.1, 1.5)

    # Initialize base PPO + structural configuration
    HYP = build_base_hyp(K)

    # Homogeneous structural values
    MU = 4.0      # mean demand
    H = 1.0       # holding cost
    B = 9.0       # backorder cost
    TBO = 10.0    # time-between-orders parameter

    # Unit batch size: LOW's action is a fraction of the remaining capacity, floored to integer units.
    bs_val = 1

    # Demand variability selection
    if cov_level == "low":
        COV = 0.14
        SIGMA = 1.0    # matches the U{3,5} demand case
    else:
        COV = 0.58
        SIGMA = 4.0    # matches the U{0,8} demand case

    # Populate CLSP instance fields
    HYP["capacity_factor"] = cf
    HYP["mu"]  = np.full(K, MU, dtype=np.float32)
    HYP["bs"]  = np.full(K, bs_val, dtype=int)
    HYP["h"]   = np.full(K, H, dtype=np.float32)
    HYP["b"]   = np.full(K, B, dtype=np.float32)
    HYP["cov"] = np.full(K, COV, dtype=np.float32)

    # Setup cost derived from classical EOQ-style relation: k_i = h_i * TBO_i^2 * μ_i / 2
    TBO_vec = np.full(K, TBO, dtype=np.float32)
    HYP["k"] = H * (TBO_vec ** 2) * HYP["mu"] / 2.0

    # No setup times in SET 1
    HYP["st"] = np.zeros(K, dtype=int)

    # Initial state (van Hezewijk et al., 2025): I_0 = ceil(mu + 1.282 * sigma), machine set up for product 0.
    init_inv_val = int(np.ceil(MU + 1.282 * SIGMA))
    HYP["init_inventory"] = np.full(K, init_inv_val, dtype=np.int32)
    HYP["init_active_idx"] = 0

    return HYP


# 4) Instance builders (S-CLSP-SD, Case II)
def build_base_hyp_sd(K):
    """Base HYP for S-CLSP-SD: as build_base_hyp plus the setup matrices st_sd and k_sd (K x K, diag = 0)."""
    HYP = build_base_hyp(K)
    HYP["st_sd"] = None
    HYP["k_sd"] = None
    return HYP


# Type A: two families by index parity.
FAM_SAME_SETUP_TIME, FAM_SAME_SETUP_COST = 0.0, 100.0
FAM_DIFF_SETUP_TIME, FAM_DIFF_SETUP_COST = 1.0, 200.0

# Type B: five families on a line, D = |f_i - f_j|. Same family: time 0, cost 100;
# different: time 0.5 * D, cost 100 + 100 * D (costs rescaled to mean 200).
FAM5_N_FAMILIES = 5
FAM5_SAME_SETUP_TIME, FAM5_SAME_SETUP_COST = 0.0, 100.0
FAM5_SETUP_TIME_PER_STEP = 0.5
FAM5_SETUP_COST_BASE, FAM5_SETUP_COST_PER_STEP = 100.0, 100.0


# Target mean of the off-diagonal setup costs (same level as the Case I setup cost k = 200)
SETUP_COST_MEAN = 200.0


def normalize_setup_cost(k_sd, target=SETUP_COST_MEAN):
    """Rescale k_sd so that the mean of its off-diagonal entries equals `target`."""
    off = ~np.eye(k_sd.shape[0], dtype=bool)
    return k_sd * (target / k_sd[off].mean())


def build_setup_matrices_families(K):
    """Type A setups: two families (even / odd index). Same family: time 0, cost 100; different: time 1,
    cost 200.
    """
    fam = np.arange(K) % 2
    same = fam[:, None] == fam[None, :]
    st = np.where(same, FAM_SAME_SETUP_TIME, FAM_DIFF_SETUP_TIME).astype(float)
    k = np.where(same, FAM_SAME_SETUP_COST, FAM_DIFF_SETUP_COST).astype(float)
    np.fill_diagonal(st, 0.0)
    np.fill_diagonal(k, 0.0)
    return k, st


def build_setup_matrices_families5(K):
    """Type B setups: five families (i mod 5) on a line, D = family distance.

    Same family: time 0, cost 100. Different: time 0.5 * D, cost 100 + 100 * D. Costs are rescaled
    afterwards.
    """
    fam = np.arange(K) % FAM5_N_FAMILIES
    D = np.abs(fam[:, None] - fam[None, :]).astype(float)
    same = D == 0
    st = np.where(same, FAM5_SAME_SETUP_TIME, FAM5_SETUP_TIME_PER_STEP * D)
    k = np.where(same, FAM5_SAME_SETUP_COST, FAM5_SETUP_COST_BASE + FAM5_SETUP_COST_PER_STEP * D)
    np.fill_diagonal(st, 0.0)
    np.fill_diagonal(k, 0.0)
    return k, st


def build_hyp_case2(sd_type, K, cov_level, cf):
    """Case II instance: the Case I setting with Type A or Type B sequence-dependent setups.

    The per-product k (EOQ cap only) is the mean incoming setup cost, as in AMBS-SD.
    """
    assert sd_type in ("A", "B")
    assert cov_level in ("low", "high")

    HYP = build_hyp_set1(K, cov_level, cf)          # Case I setting
    HYP["st_sd"] = None
    HYP["k_sd"] = None

    if sd_type == "A":
        k_sd, st_sd = build_setup_matrices_families(K)
    else:
        k_sd, st_sd = build_setup_matrices_families5(K)
    k_sd = normalize_setup_cost(k_sd)               # mean off-diagonal setup cost = 200
    HYP["k_sd"] = k_sd
    HYP["st_sd"] = st_sd

    off = ~np.eye(K, dtype=bool)
    HYP["k"] = np.array([k_sd[:, i][off[:, i]].mean() for i in range(K)], dtype=np.float32)
    HYP["st"] = np.zeros(K, dtype=int)               # unused when st_sd is given
    return HYP


# 5) Paper instance registry (ID -> builder parameters)
CASE_I_REGISTRY = {
    # K = 5
    "1.1":  dict(K=5,  cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.2":  dict(K=5,  cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.3":  dict(K=5,  cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.4":  dict(K=5,  cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 10
    "1.5":  dict(K=10, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.6":  dict(K=10, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.7":  dict(K=10, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.8":  dict(K=10, cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 15
    "1.9":  dict(K=15, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.10": dict(K=15, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.11": dict(K=15, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.12": dict(K=15, cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 20
    "1.13": dict(K=20, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.14": dict(K=20, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.15": dict(K=20, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.16": dict(K=20, cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 25
    "1.17": dict(K=25, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.18": dict(K=25, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.19": dict(K=25, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.20": dict(K=25, cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 30
    "1.21": dict(K=30, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.22": dict(K=30, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.23": dict(K=30, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.24": dict(K=30, cf=1.5, cov="high"),  # D_i = U{0,8}
}


# Case II: Case I setting with sequence-dependent setups (Type A: 2.1-2.8, Type B: 2.9-2.16).
CASE_II_REGISTRY = {
    "2.1":  dict(sd_type="A", K=15, cf=1.1, cov="low"),   # D_i = U{3,5}
    "2.2":  dict(sd_type="A", K=15, cf=1.5, cov="low"),   # D_i = U{3,5}
    "2.3":  dict(sd_type="A", K=15, cf=1.1, cov="high"),  # D_i = U{0,8}
    "2.4":  dict(sd_type="A", K=15, cf=1.5, cov="high"),  # D_i = U{0,8}
    "2.5":  dict(sd_type="A", K=20, cf=1.1, cov="low"),   # D_i = U{3,5}
    "2.6":  dict(sd_type="A", K=20, cf=1.5, cov="low"),   # D_i = U{3,5}
    "2.7":  dict(sd_type="A", K=20, cf=1.1, cov="high"),  # D_i = U{0,8}
    "2.8":  dict(sd_type="A", K=20, cf=1.5, cov="high"),  # D_i = U{0,8}
    "2.9":  dict(sd_type="B", K=15, cf=1.1, cov="low"),   # D_i = U{3,5}
    "2.10": dict(sd_type="B", K=15, cf=1.5, cov="low"),   # D_i = U{3,5}
    "2.11": dict(sd_type="B", K=15, cf=1.1, cov="high"),  # D_i = U{0,8}
    "2.12": dict(sd_type="B", K=15, cf=1.5, cov="high"),  # D_i = U{0,8}
    "2.13": dict(sd_type="B", K=20, cf=1.1, cov="low"),   # D_i = U{3,5}
    "2.14": dict(sd_type="B", K=20, cf=1.5, cov="low"),   # D_i = U{3,5}
    "2.15": dict(sd_type="B", K=20, cf=1.1, cov="high"),  # D_i = U{0,8}
    "2.16": dict(sd_type="B", K=20, cf=1.5, cov="high"),  # D_i = U{0,8}
}


def resolve_case1_env_helpers(K):
    """Environment helper flags for an instance with K products, applying the ENV_* overrides.


    :return: Dict with the five boolean flags.
    """
    if K <= 10:
        defaults = dict(
            reserve_future_min=False,
            force_min_batch=False,
            enable_Ilim=True,
            enforce_eoq_qmax=False,
            enable_ip_eligibility=True,
        )
    else:
        defaults = dict(
            reserve_future_min=True,
            force_min_batch=True,
            enable_Ilim=True,
            enforce_eoq_qmax=True,
            enable_ip_eligibility=True,
        )

    overrides = dict(
        reserve_future_min=ENV_RESERVE_FUTURE_MIN,
        force_min_batch=ENV_FORCE_MIN_BATCH,
        enable_Ilim=ENV_ENABLE_ILIM,
        enforce_eoq_qmax=ENV_ENFORCE_EOQ_QMAX,
        enable_ip_eligibility=ENV_ENABLE_IP_ELIGIBILITY,
    )

    return {
        flag: (defaults[flag] if override is None else bool(override))
        for flag, override in overrides.items()
    }


def build_hyp_from_instance_id(instance_id):
    instance_id = instance_id.strip()

    # Case I (S-CLSP) — homogeneous demand suite
    if instance_id in CASE_I_REGISTRY:
        cfg = CASE_I_REGISTRY[instance_id]
        HYP = build_hyp_set1(cfg["K"], cfg["cov"], cfg["cf"])
        HYP.update(resolve_case1_env_helpers(cfg["K"]))
        problem = "SCLSP"

        exp_name = f"Case_1_ID_{instance_id}_K{cfg['K']}_cov{cfg['cov']}_cf{cfg['cf']}"
        return problem, HYP, exp_name

    # Case II (S-CLSP-SD) — sequence-dependent setups
    if instance_id in CASE_II_REGISTRY:
        cfg = CASE_II_REGISTRY[instance_id]
        HYP = build_hyp_case2(cfg["sd_type"], cfg["K"], cfg["cov"], cfg["cf"])
        HYP.update(resolve_case1_env_helpers(cfg["K"]))
        problem = "SCLSP_SD"

        exp_name = (f"Case_2_ID_{instance_id}_type{cfg['sd_type']}"
                    f"_K{cfg['K']}_cov{cfg['cov']}_cf{cfg['cf']}")
        return problem, HYP, exp_name

    raise ValueError(f"Unknown INSTANCE_ID '{instance_id}'. Not found in paper registry.")


# 6) Entry point: build the instance and the hyperparameters (HYP)
PROBLEM, HYP, exp_name = build_hyp_from_instance_id(INSTANCE_ID)

# Outputs of this experiment go to ./runs/<exp_name>.
HYP["out_dir"] = os.path.join("runs", exp_name)
os.makedirs(HYP["out_dir"], exist_ok=True)

# Optional network-size override (see HIDDEN_SIZES* above)
if HIDDEN_SIZES is not None:
    HYP["hidden_sizes"] = tuple(HIDDEN_SIZES)
if HIDDEN_SIZES_HIGH is not None:
    HYP["hidden_high"] = tuple(HIDDEN_SIZES_HIGH)
if HIDDEN_SIZES_LOW is not None:
    HYP["hidden_low"] = tuple(HIDDEN_SIZES_LOW)

# The warm-start cache (high_warmstart.pt) is stored in this run's own folder.
HYP["warm_start_dir"] = HYP["out_dir"]
HYP["warm_start"] = WARM_START
HYP["warm_start_force_retrain"] = WARM_START_FORCE_RETRAIN
HYP["instance_id"] = INSTANCE_ID
HYP["warm_start_kwargs"] = dict(WARM_START_KWARGS)

# LOW return definition and per-level discount factors (see global selector above)
HYP["low_return_horizon"] = int(LOW_RETURN_H)
HYP["low_succ_horizon"] = int(LOW_SUCC_H)
HYP["low_reward_scale"] = float(LOW_REWARD_SCALE)
print(f"[config] LOW return H={HYP['low_return_horizon']} | "
      f"succ_H={HYP['low_succ_horizon']} | gamma_high={HYP.get('gamma_high', HYP['gamma'])} | gamma_low={HYP.get('gamma_low', HYP['gamma'])} | "
      f"entropy_coef_high={HYP.get('entropy_coef_high', HYP['entropy_coef'])} | entropy_coef_low={HYP.get('entropy_coef_low', HYP['entropy_coef'])} | "
      f"scale={HYP['low_reward_scale']}")


## 2. Train

Skip this cell to test an already trained model.


In [ ]:
"""
Train HIGH + LOW with PPO on the instance defined in the previous cell.
Skip this cell if you only want to test an already trained model.
"""

# 7) Training: S-CLSP
def run_sclsp_with_hyp(HYP, exp_name, SEED=123):
    """Train the hierarchical PPO on a Case I (S-CLSP) instance, with optional warm start."""
    # 0) Global configuration
    set_global_seeds(SEED)

    # Automatically select GPU if available
    DEVICE = get_device()

    # Optional experiment tracking (activated via HYP["use_wandb"])
    import wandb

    # Environment factory and hierarchical PPO implementation
    from CLSP_HDRL import make_env_from_hyp, PPO_Hier

    # 1) WandB (optional)
    if HYP.get("use_wandb", False):
        wandb.login()
        run = wandb.init(project="Pruebas Arquitectura", config=HYP, name=exp_name)
    else:
        run = None

    # 2) Environment and agent initialization
    env = make_env_from_hyp(HYP, seed=SEED)
    trainer = PPO_Hier(env, HYP, device=DEVICE, seed=SEED)

    # 2b) Optional AMBS-based warm start (HIGH)
    if HYP.get("warm_start", False):
        from CLSP_HDRL import warm_start_policies
        # Read/write the warm-start cache (high_warmstart.pt) in HYP["warm_start_dir"], then restore out_dir.
        _training_out_dir = HYP["out_dir"]
        HYP["out_dir"] = HYP.get("warm_start_dir", _training_out_dir)
        warm_start_policies(
            trainer, HYP, instance_id=HYP.get("instance_id"),
            device=DEVICE,
            seed=SEED,
            warm_start_kwargs=HYP.get("warm_start_kwargs"),
        )
        HYP["out_dir"] = _training_out_dir

    # 3) Training
    trainer.train()
    if run is not None:
        run.finish()


# 7b) Training: S-CLSP-SD
def run_sclsp_sd_with_hyp(HYP, exp_name, SEED=123):
    """Same as run_sclsp_with_hyp for Case II (S-CLSP-SD): CLSPHierEnv2 and PPO_Hier2."""
    # 0) Global configuration
    set_global_seeds(SEED)

    # Automatically select GPU if available
    DEVICE = get_device()

    # Optional experiment tracking (activated via HYP["use_wandb"])
    import wandb

    # Environment factory and hierarchical PPO implementation
    from CLSP_HDRL import make_env_from_hyp2, PPO_Hier2

    # 1) WandB (optional)
    if HYP.get("use_wandb", False):
        wandb.login()
        run = wandb.init(project="Pruebas Arquitectura", config=HYP, name=exp_name)
    else:
        run = None

    # 2) Environment and agent initialization
    env = make_env_from_hyp2(HYP, seed=SEED)
    trainer = PPO_Hier2(env, HYP, device=DEVICE, seed=SEED)

    # 2b) Optional AMBS-based warm start (HIGH)
    if HYP.get("warm_start", False):
        from CLSP_HDRL import warm_start_policies
        # Read/write the warm-start cache (high_warmstart.pt) in HYP["warm_start_dir"], then restore out_dir.
        _training_out_dir = HYP["out_dir"]
        HYP["out_dir"] = HYP.get("warm_start_dir", _training_out_dir)
        warm_start_policies(
            trainer, HYP, instance_id=HYP.get("instance_id"),
            device=DEVICE,
            seed=SEED,
            warm_start_kwargs=HYP.get("warm_start_kwargs"),
        )
        HYP["out_dir"] = _training_out_dir

    # 3) Training
    trainer.train()
    if run is not None:
        run.finish()


# 8) Launch training
if PROBLEM == "SCLSP":
    run_sclsp_with_hyp(HYP, exp_name, SEED=123)
elif PROBLEM == "SCLSP_SD":
    run_sclsp_sd_with_hyp(HYP, exp_name, SEED=123)
else:
    raise ValueError('PROBLEM must be "SCLSP" or "SCLSP_SD".')


## 3. Test

Loads `high_joint_best.pt` / `low_joint_best.pt` from the folder of this notebook.


In [ ]:
"""
Test a trained HIGH + LOW pair on the instance defined in the instance cell.

Run the instance cell first (it sets INSTANCE_ID and HYP). The training cell can be skipped.
The models are read from the folder of this notebook: just put high_joint_best.pt and low_joint_best.pt there,
no path needed. If they are not there, the ones just saved by the training cell of this session
(HYP["out_dir"]) are used.
"""
# -----------------------
# 0) Global configuration
# -----------------------
# Seed for run reproducibility (net init, rollouts, demand sampling, etc.)
SEED = 123
set_global_seeds(SEED)
DEVICE = "cpu"

# -----------------------
# 1) Trained models
# -----------------------
# Same folder as this notebook (current working directory of the notebook).
MODEL_DIR = os.getcwd()
ckpt_high = os.path.join(MODEL_DIR, "high_joint_best.pt")
ckpt_low = os.path.join(MODEL_DIR, "low_joint_best.pt")

if not (os.path.isfile(ckpt_high) and os.path.isfile(ckpt_low)):
    # Fallback: models saved by the training cell of this session
    ckpt_high = os.path.join(HYP["out_dir"], "high_joint_best.pt")
    ckpt_low = os.path.join(HYP["out_dir"], "low_joint_best.pt")

if not (os.path.isfile(ckpt_high) and os.path.isfile(ckpt_low)):
    raise FileNotFoundError(
        "high_joint_best.pt / low_joint_best.pt not found in the folder of this notebook "
        "(nor in HYP['out_dir']). Put the two files of the trained model there and run again."
    )
print("HIGH model:", ckpt_high)
print("LOW model: ", ckpt_low)

# -----------------------
# 2) Evaluation
# -----------------------
# Environment factory and hierarchical PPO implementation (S-CLSP or S-CLSP-SD)
if PROBLEM == "SCLSP_SD":
    from CLSP_HDRL import make_env_from_hyp2 as make_env, PPO_Hier2 as Trainer
else:
    from CLSP_HDRL import make_env_from_hyp as make_env, PPO_Hier as Trainer

env = make_env(HYP, seed=SEED)
trainer = Trainer(env, HYP, device=DEVICE, seed=SEED)

res = trainer.test(
    n_runs=10_000,
    warmup=128,
    greedy=True,
    seed_base=12,
    ckpt_high=ckpt_high,
    ckpt_low=ckpt_low,
    env_overrides={"horizon": 228, "enable_Ilim": False},
)

print("==============================================")
print(f"Experiment: {exp_name}")
print("Mean cost over runs:", res["mean_of_means"])
print("==============================================")
